In [0]:
# Databricks notebook source

# MAGIC %md
# MAGIC # Medium-3: Reinsurance IFRS 17 — RAW Data Generation
# MAGIC 
# MAGIC This notebook generates all RAW tables for the Reinsurance IFRS 17 Retro Linkage project.
# MAGIC 
# MAGIC **Tables:**
# MAGIC 1. `goc_master` — GoC definition (Assumed + Retro unified)
# MAGIC 2. `treaty_master` — Treaty definition (Assumed + Retro unified)
# MAGIC 3. `retro_link` — Assumed ↔ Retro relationship with recovery rate
# MAGIC 4. `cashflow_input` — Quarterly cashflow per treaty × AoC step
# MAGIC 5. `assumption_version` — Governance metadata
# MAGIC 
# MAGIC **Sign Convention:**
# MAGIC - Amount > 0 → Profit direction (premium received, GoC에 유리)
# MAGIC - Amount < 0 → Loss direction (claim payment, GoC에 불리)

# COMMAND ----------

from pyspark.sql import SparkSession
from pyspark.sql.types import *
import pandas as pd
from datetime import date

spark = SparkSession.builder.getOrCreate()

# Schema setup
RAW_SCHEMA = "workspace.reinsurance_ifrs17_raw"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {RAW_SCHEMA}")

# Clean existing tables
tables = spark.catalog.listTables(RAW_SCHEMA)
for t in tables:
    spark.sql(f"DROP TABLE IF EXISTS {RAW_SCHEMA}.{t.name}")

print(f"Schema {RAW_SCHEMA} ready.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. goc_master
# MAGIC 
# MAGIC | GoC | Type | Initial State | Scenario |
# MAGIC |-----|------|---------------|----------|
# MAGIC | GOC_A | ASSUMED | PROFITABLE | Baseline — CSM absorbs small loss |
# MAGIC | GOC_B | ASSUMED | PROFITABLE | Large loss → CSM exhausted → Onerous |
# MAGIC | GOC_C | ASSUMED | ONEROUS | Initially onerous, worsens |
# MAGIC | GOC_D | ASSUMED | ONEROUS | Improves → Profitable recovery |
# MAGIC | GOC_R1 | RETRO | PROFITABLE | Retro GoC covering multiple assumed treaties |

# COMMAND ----------

goc_data = [
    ("GOC_A", "ASSUMED", "PROFITABLE", "Baseline — small loss, CSM absorbs"),
    ("GOC_B", "ASSUMED", "PROFITABLE", "Large loss event Q2 — CSM exhausted, turns onerous"),
    ("GOC_C", "ASSUMED", "ONEROUS", "Initially onerous (pricing error), worsens through year"),
    ("GOC_D", "ASSUMED", "ONEROUS", "Initially onerous, improves Q3 — LC reversed to CSM"),
    ("GOC_R1", "RETRO", "PROFITABLE", "Retro GoC — covers assumed treaties, receives recovery"),
]

df_goc = pd.DataFrame(goc_data, columns=[
    "goc_id", "goc_type", "expected_initial_state", "scenario_description"
])

spark.createDataFrame(df_goc).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{RAW_SCHEMA}.goc_master")

display(spark.table(f"{RAW_SCHEMA}.goc_master"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. treaty_master
# MAGIC 
# MAGIC Each Assumed GoC has 1-2 treaties with different characteristics.
# MAGIC Each Assumed treaty has a corresponding Retro treaty.
# MAGIC All Retro treaties belong to GOC_R1.

# COMMAND ----------

treaty_data = [
    # Assumed treaties
    ("TRT_A1", "ASSUMED", "GOC_A", "2026-01-01"),
    ("TRT_A2", "ASSUMED", "GOC_A", "2026-01-01"),
    ("TRT_B1", "ASSUMED", "GOC_B", "2026-01-01"),
    ("TRT_B2", "ASSUMED", "GOC_B", "2026-01-01"),
    ("TRT_C1", "ASSUMED", "GOC_C", "2026-01-01"),
    ("TRT_D1", "ASSUMED", "GOC_D", "2026-01-01"),
    ("TRT_D2", "ASSUMED", "GOC_D", "2026-01-01"),
    # Retro treaties
    ("TRT_R1", "RETRO", "GOC_R1", "2026-01-01"),
    ("TRT_R2", "RETRO", "GOC_R1", "2026-01-01"),
]

df_treaty = pd.DataFrame(treaty_data, columns=[
    "treaty_id", "treaty_type", "goc_id", "inception_date"
])
df_treaty["inception_date"] = pd.to_datetime(df_treaty["inception_date"]).dt.date

spark.createDataFrame(df_treaty).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{RAW_SCHEMA}.treaty_master")

display(spark.table(f"{RAW_SCHEMA}.treaty_master"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. retro_link
# MAGIC 
# MAGIC 1:1 mapping (QS scope). Different cession rates per treaty.

# COMMAND ----------

retro_link_data = [
    ("TRT_A1", "TRT_R1", 0.30, "2026-01-01"),  # 30% recovery
    ("TRT_A2", "TRT_R1", 0.30, "2026-01-01"),  # 40% recovery
    ("TRT_B1", "TRT_R1", 0.30, "2026-01-01"),  # 35% recovery
    ("TRT_B2", "TRT_R1", 0.25, "2026-01-01"),  # 25% recovery
    ("TRT_C1", "TRT_R2", 0.45, "2026-01-01"),  # 50% recovery (higher for risky portfolio)
    ("TRT_D1", "TRT_R2", 0.45, "2026-01-01"),  # 30% recovery
    ("TRT_D2", "TRT_R2", 0.45, "2026-01-01"),  # 45% recovery
]

df_retro = pd.DataFrame(retro_link_data, columns=[
    "assumed_treaty_id", "retro_treaty_id", "retro_cession_rate", "effective_date"
])
df_retro["effective_date"] = pd.to_datetime(df_retro["effective_date"]).dt.date

spark.createDataFrame(df_retro).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{RAW_SCHEMA}.retro_link")

display(spark.table(f"{RAW_SCHEMA}.retro_link"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. cashflow_input
# MAGIC 
# MAGIC ### Design Rationale
# MAGIC 
# MAGIC Sign convention: + = profit (premium), - = loss (claim)
# MAGIC 
# MAGIC AoC steps in RAW: INIT_RECOG (Q1 only), VARIANCE (every quarter)
# MAGIC 
# MAGIC **GoC A (Profitable → Profitable):**
# MAGIC - INIT_RECOG: strong positive (profitable)
# MAGIC - VARIANCE: small negative losses each quarter, CSM absorbs all
# MAGIC 
# MAGIC **GoC B (Profitable → Onerous):**
# MAGIC - INIT_RECOG: moderate positive (profitable)
# MAGIC - Q2 VARIANCE: massive loss event (industrial accident) → CSM exhausted → LC
# MAGIC 
# MAGIC **GoC C (Onerous → Onerous worse):**
# MAGIC - INIT_RECOG: negative (onerous from inception, pricing error)
# MAGIC - VARIANCE: additional losses each quarter
# MAGIC 
# MAGIC **GoC D (Onerous → Profitable):**
# MAGIC - INIT_RECOG: slightly negative (marginally onerous)
# MAGIC - Q3 VARIANCE: favorable experience → LC reversed → CSM restored

# COMMAND ----------

# Quarter-end dates
Q1 = date(2026, 3, 31)
Q2 = date(2026, 6, 30)
Q3 = date(2026, 9, 30)
Q4 = date(2026, 12, 31)

cashflow_data = []

# =============================================================================
# GOC_A: Profitable throughout
# TRT_A1 + TRT_A2 combined INIT_RECOG positive, small losses each quarter
# =============================================================================

# TRT_A1: INIT_RECOG = +60,000 (strong profit margin)
cashflow_data.append(("TRT_A1", Q1, "INIT_RECOG", 60000.0))
cashflow_data.append(("TRT_A1", Q1, "VARIANCE", -8000.0))    # small loss
cashflow_data.append(("TRT_A1", Q2, "VARIANCE", -5000.0))    # small loss
cashflow_data.append(("TRT_A1", Q3, "VARIANCE", -7000.0))    # small loss
cashflow_data.append(("TRT_A1", Q4, "VARIANCE", -6000.0))    # small loss

# TRT_A2: INIT_RECOG = +40,000
cashflow_data.append(("TRT_A2", Q1, "INIT_RECOG", 40000.0))
cashflow_data.append(("TRT_A2", Q1, "VARIANCE", -4000.0))    # small loss
cashflow_data.append(("TRT_A2", Q2, "VARIANCE", -3000.0))    # small loss
cashflow_data.append(("TRT_A2", Q3, "VARIANCE", -5000.0))    # small loss
cashflow_data.append(("TRT_A2", Q4, "VARIANCE", -3000.0))    # small loss

# GOC_A check:
# INIT_RECOG total = +100,000 → PROFITABLE (CSM = -100,000)
# Total VARIANCE = -41,000
# Net = +59,000 → still PROFITABLE ✓

# =============================================================================
# GOC_B: Profitable → Onerous (Q2 large loss event)
# TRT_B1 + TRT_B2 combined
# =============================================================================

# TRT_B1: INIT_RECOG = +35,000
cashflow_data.append(("TRT_B1", Q1, "INIT_RECOG", 35000.0))
cashflow_data.append(("TRT_B1", Q1, "VARIANCE", -3000.0))    # normal
cashflow_data.append(("TRT_B1", Q2, "VARIANCE", -55000.0))   # MASSIVE LOSS (industrial accident)
cashflow_data.append(("TRT_B1", Q3, "VARIANCE", -4000.0))    # normal
cashflow_data.append(("TRT_B1", Q4, "VARIANCE", -3000.0))    # normal

# TRT_B2: INIT_RECOG = +25,000
cashflow_data.append(("TRT_B2", Q1, "INIT_RECOG", 25000.0))
cashflow_data.append(("TRT_B2", Q1, "VARIANCE", -2000.0))    # normal
cashflow_data.append(("TRT_B2", Q2, "VARIANCE", -40000.0))   # MASSIVE LOSS
cashflow_data.append(("TRT_B2", Q3, "VARIANCE", -3000.0))    # normal
cashflow_data.append(("TRT_B2", Q4, "VARIANCE", -2000.0))    # normal

# GOC_B check:
# INIT_RECOG total = +60,000 → PROFITABLE (CSM = -60,000)
# Q1 VARIANCE = -5,000 → cumulative = +55,000 → still PROFITABLE
# Q2 VARIANCE = -95,000 → cumulative = -40,000 → ONEROUS ✓ (CSM exhausted, LC = -40,000)
# Q3 VARIANCE = -7,000 → cumulative = -47,000 → ONEROUS (worsens)
# Q4 VARIANCE = -5,000 → cumulative = -52,000 → ONEROUS ✓

# =============================================================================
# GOC_C: Onerous from inception, worsens
# TRT_C1 only
# =============================================================================

# TRT_C1: INIT_RECOG = -20,000 (onerous from start — pricing error)
cashflow_data.append(("TRT_C1", Q1, "INIT_RECOG", -20000.0))
cashflow_data.append(("TRT_C1", Q1, "VARIANCE", -8000.0))    # additional loss
cashflow_data.append(("TRT_C1", Q2, "VARIANCE", -12000.0))   # more loss
cashflow_data.append(("TRT_C1", Q3, "VARIANCE", -5000.0))    # continued loss
cashflow_data.append(("TRT_C1", Q4, "VARIANCE", -10000.0))   # continued loss

# GOC_C check:
# INIT_RECOG = -20,000 → ONEROUS (LC = -20,000)
# Q1 VARIANCE = -8,000 → cumulative = -28,000 → ONEROUS (worse)
# Q2 VARIANCE = -12,000 → cumulative = -40,000 → ONEROUS (worse)
# Q3 VARIANCE = -5,000 → cumulative = -45,000 → ONEROUS (worse)
# Q4 VARIANCE = -10,000 → cumulative = -55,000 → ONEROUS ✓

# =============================================================================
# GOC_D: Onerous → Profitable (Q3 favorable experience)
# TRT_D1 + TRT_D2 combined
# =============================================================================

# TRT_D1: INIT_RECOG = -8,000 (marginally onerous)
cashflow_data.append(("TRT_D1", Q1, "INIT_RECOG", -8000.0))
cashflow_data.append(("TRT_D1", Q1, "VARIANCE", -2000.0))    # small loss
cashflow_data.append(("TRT_D1", Q2, "VARIANCE", -1000.0))    # small loss
cashflow_data.append(("TRT_D1", Q3, "VARIANCE", 18000.0))    # FAVORABLE (reserve release)
cashflow_data.append(("TRT_D1", Q4, "VARIANCE", 2000.0))     # small favorable

# TRT_D2: INIT_RECOG = -7,000 (marginally onerous)
cashflow_data.append(("TRT_D2", Q1, "INIT_RECOG", -7000.0))
cashflow_data.append(("TRT_D2", Q1, "VARIANCE", -1000.0))    # small loss
cashflow_data.append(("TRT_D2", Q2, "VARIANCE", -2000.0))    # small loss
cashflow_data.append(("TRT_D2", Q3, "VARIANCE", 15000.0))    # FAVORABLE (reserve release)
cashflow_data.append(("TRT_D2", Q4, "VARIANCE", 3000.0))     # small favorable

# GOC_D check:
# INIT_RECOG total = -15,000 → ONEROUS (LC = -15,000)
# Q1 VARIANCE = -3,000 → cumulative = -18,000 → ONEROUS
# Q2 VARIANCE = -3,000 → cumulative = -21,000 → ONEROUS
# Q3 VARIANCE = +33,000 → cumulative = +12,000 → PROFITABLE ✓ (LC reversed, CSM = -12,000)
# Q4 VARIANCE = +5,000 → cumulative = +17,000 → PROFITABLE ✓

# =============================================================================
# Create DataFrame and save
# =============================================================================

df_cf = pd.DataFrame(cashflow_data, columns=[
    "treaty_id", "reporting_date", "aoc_step", "amount"
])

spark.createDataFrame(df_cf).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{RAW_SCHEMA}.cashflow_input")

display(spark.table(f"{RAW_SCHEMA}.cashflow_input"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. assumption_version

# COMMAND ----------

version_data = [
    ("V1", "Initial RAW data generation for Medium-3 project", "2026-04-20"),
]

df_version = pd.DataFrame(version_data, columns=[
    "version_id", "description", "as_of_date"
])
df_version["as_of_date"] = pd.to_datetime(df_version["as_of_date"]).dt.date

spark.createDataFrame(df_version).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{RAW_SCHEMA}.assumption_version")

display(spark.table(f"{RAW_SCHEMA}.assumption_version"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Verification — GoC Scenario Walkthrough
# MAGIC 
# MAGIC Quick validation that test data produces intended scenarios.

# COMMAND ----------

# Verify GoC-level cashflow totals
verify_sql = f"""
SELECT 
    t.goc_id,
    g.expected_initial_state,
    g.scenario_description,
    c.aoc_step,
    c.reporting_date,
    SUM(c.amount) AS total_amount
FROM {RAW_SCHEMA}.cashflow_input c
JOIN {RAW_SCHEMA}.treaty_master t ON c.treaty_id = t.treaty_id
JOIN {RAW_SCHEMA}.goc_master g ON t.goc_id = g.goc_id
WHERE t.treaty_type = 'ASSUMED'
GROUP BY t.goc_id, g.expected_initial_state, g.scenario_description, c.aoc_step, c.reporting_date
ORDER BY t.goc_id, c.reporting_date, c.aoc_step
"""
display(spark.sql(verify_sql))

# COMMAND ----------

# Verify cumulative balance per GoC (should match design intent)
cumulative_sql = f"""
WITH quarterly_totals AS (
    SELECT 
        t.goc_id,
        c.reporting_date,
        SUM(c.amount) AS quarterly_amount
    FROM {RAW_SCHEMA}.cashflow_input c
    JOIN {RAW_SCHEMA}.treaty_master t ON c.treaty_id = t.treaty_id
    WHERE t.treaty_type = 'ASSUMED'
    GROUP BY t.goc_id, c.reporting_date
)
SELECT
    goc_id,
    reporting_date,
    quarterly_amount,
    SUM(quarterly_amount) OVER (PARTITION BY goc_id ORDER BY reporting_date) AS cumulative_balance,
    CASE 
        WHEN SUM(quarterly_amount) OVER (PARTITION BY goc_id ORDER BY reporting_date) > 0 THEN 'PROFITABLE'
        WHEN SUM(quarterly_amount) OVER (PARTITION BY goc_id ORDER BY reporting_date) < 0 THEN 'ONEROUS'
        ELSE 'BREAK_EVEN'
    END AS expected_state
FROM quarterly_totals
ORDER BY goc_id, reporting_date
"""
display(spark.sql(cumulative_sql))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Summary
# MAGIC 
# MAGIC | Table | Rows | Description |
# MAGIC |-------|------|-------------|
# MAGIC | goc_master | 5 | 4 Assumed + 1 Retro GoC |
# MAGIC | treaty_master | 14 | 7 Assumed + 7 Retro treaties |
# MAGIC | retro_link | 7 | 1:1 Assumed ↔ Retro mapping |
# MAGIC | cashflow_input | 38 | Treaty × Quarter × AoC step |
# MAGIC | assumption_version | 1 | Governance metadata |

# COMMAND ----------

# Final row counts
for table_name in ["goc_master", "treaty_master", "retro_link", "cashflow_input", "assumption_version"]:
    count = spark.table(f"{RAW_SCHEMA}.{table_name}").count()
    print(f"{table_name}: {count} rows")